
## 1. What is Structured Streaming?


- **Structured Streaming** is a `Spark feature` used to `process continuously arriving data in real time`.
- It processes `incoming data incrementally` using the `DataFrame and SQL APIs`.


### Real-time example:


**Consider an e-commerce application like Amazon.**

1. Customers place orders continuously.
2. Orders arrive from Kafka.
3. Spark reads incoming orders.
4. Invalid orders are filtered.
5. Valid orders are processed.
6. Sales data is stored in Delta Lake.


## 2. What are Sources and Sinks?


**Source:**

 A **source** is where `streaming data comes from`.

**Sink:** 

A **sink** is where `processed streaming data is stored`.


| Sources | Sinks |
|---|---|
| Kafka | Delta Lake |
| Files (CSV, JSON, Parquet) | Console |
| Auto Loader | Kafka |
| Rate Source | Memory |
| Socket | Files |


## Real-time example:


**An e-commerce application receives orders from Kafka and stores processed orders in Delta Lake.**



```text
+-----------------------------+
|       SOURCE: KAFKA         |
|                             |
|   Incoming Customer Orders  |
+-----------------------------+
              |
              v
+-----------------------------+
| SPARK STRUCTURED STREAMING  |
|                             |
|  1. Filter Invalid Records  |
|  2. Clean Data              |
|  3. Transform Data           |
+-----------------------------+
              |
              v
+-----------------------------+
|       SINK: DELTA LAKE      |
|                             |
|   Store Processed Orders    |
+-----------------------------+


## 3. What is a Trigger in Structured Streaming?


A **trigger** in Structured Streaming `controls when Spark processes incoming streaming data`.



### Example:


- ProcessingTime with a 10-second interval processes data every 10 seconds. 
- Other trigger types include Once, AvailableNow, Default and Continuous. We choose a trigger based on latency, processing requirements and cost.


## Types of Triggers


| Trigger Type | Explanation |
|---|---|
| Default Trigger | Processes data as soon as possible using micro-batches. |
| ProcessingTime | Processes data at a specified time interval. |
| Once | Processes all available data in a single batch and stops. |
| AvailableNow | Processes all currently available data in multiple batches and stops. |
| Continuous | Processes records continuously with low latency (experimental). |


### Real-time example:


**A banking application processes transactions every 10 seconds.**

In [0]:
query = (
    orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start(output_path)
)


### 4. What are Output Modes?


**Output Modes** determine `how the results of a streaming query are written to the output sink`.


### Types of Output Modes:


**1. Append Mode**

- Writes only `newly finalized row`s.
- Previously written rows are not updated.
- Suitable for new customer orders.

**2. Complete Mode**

- Writes the` entire current result table`.
- All aggregation results are output after every trigger.
- Suitable for total sales by region.

**3. Update Mode**

- Writes `only rows whose values have changed`.
- Unchanged rows are not written again.
- Suitable for continuously updated sales totals.


### Real-Time Example:


**Suppose we calculate total sales by region.**

| Region | Total Sales |
|---|---:|
| Chennai | 50,000 |
| Bangalore | 75,000 |


## 5. What is a Watermark?


- A **Watermark** in Structured Streaming determines `how long Spark should wait for late-arriving data`,
- Before considering an `aggregation window complete and cleaning up old state`.


**An order is created at 10:00 AM, but due to network delay, it reaches Spark at 10:15 AM.**

In [0]:
df = df.withWatermark("order_time", "10 minutes")


## Why do we use watermarks?


- Handle late-arriving data.
- Control state-store size.
- Prevent unlimited state growth.
- Support event-time window aggregations.


## 6. What are Windows in Structured Streaming?


**Windows** is a technique used to `group streaming records based on a time interval and perform aggregations` such as `count, sum, and average`.


### Types of Windows:


| Window Type | Explanation | Real-Time Example |
|---|---|---|
| Tumbling Window | Fixed-size, non-overlapping windows. | Calculate sales every 5 minutes. |
| Sliding Window | Fixed-size windows that overlap. | Calculate sales for the last 10 minutes, updating every 5 minutes. |
| Session Window | Groups events based on periods of activity separated by inactivity gaps. | Group customer clicks until the customer is inactive for 10 minutes. |


### Real-Time Example:


**An e-commerce company wants to calculate total sales every 5 minutes.**

| Time Window | Total Sales |
|---|---:|
| 10:00–10:05 | ₹50,000 |
| 10:05–10:10 | ₹75,000 |
| 10:10–10:15 | ₹90,000 |


### Example – Tumbling Window:

In [0]:
from pyspark.sql.functions import *

result = df \
    .withWatermark("order_time", "10 minutes") \
    .groupBy(
        window("order_time", "5 minutes")
    ) \
    .agg(
        sum("amount").alias("total_sales")
    )


## 7. What is Checkpointing?


- **Checkpointing** is a `fault-tolerance` mechanism used to stores `streaming query progress and state information` in durable storage. 
- If a failure occurs, Spark can `recover and continue processing` using the checkpoint information.


### Real-time example:


**An e-commerce streaming application processes 10,000 orders. Suddenly, the Spark cluster crashes.**

**After restarting the query with the same checkpoint location, Spark can recover its progress and state instead of starting from scratch.**

In [0]:
query = df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/checkpoints/orders"
    ) \
    .start("/delta/orders")


## Why do we use checkpointing?


- Recover from failures.
- Track processed offsets.
- Maintain streaming state.
- Support fault-tolerant processing.


## 8. What is Kafka Integration with Structured Streaming?


- **Kafka integration** allows **Spark Structured Streaming** to `consume real-time messages from Kafka topics`,
- ` process them using Spark transformations`, 
- `write the results to destinations such as Delta Lake`.


### Real-time example:


**A banking application publishes transaction messages to a Kafka topic. Spark reads those transactions, identifies suspicious transactions, and stores the processed records in Delta Lake for further analysis.**


![image_1793702368798.png](./image_1793702368798.png "image_1793702368798.png")


### PySpark – Read from Kafka:

In [0]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    ) \
    .option("subscribe", "transactions") \
    .load()


### Convert Kafka value into readable text:

In [0]:
from pyspark.sql.functions import *

orders_df = kafka_df.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("timestamp")
)


### Write to Delta Lake:

In [0]:
query = orders_df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/checkpoints/transactions"
    ) \
    .start("/delta/transactions")



## Important Kafka Columns in Spark Structured Streaming:


| Column | Explanation | Example |
|---|---|---|
| key | Message key used to identify or partition messages. | customer_101 |
| value | Actual message data (payload). | {"customer_id":101,"amount":5000} |
| topic | Kafka topic name from which the message is received. | customer_orders |
| partition | Partition number where the message is stored. | 0 |
| offset | Unique sequential position of a message within a partition. | 125 |
| timestamp | Timestamp associated with the Kafka message. | 2026-11-03 10:30:00 |



### Real-Time Example:


**An e-commerce company sends customer orders to Kafka.**

| Column | Value |
|---|---|
| key | customer_101 |
| value | {"order_id":501,"amount":5000} |
| topic | customer_orders |
| partition | 0 |
| offset | 125 |
| timestamp | 2026-11-03 10:30:00 |


### 9. What is Auto Loader in Databricks?


**Auto Loader** is a `Databricks feature` used to `automatically and incrementally load new files from cloud storage into Delta Lake using Spark Structured Streaming`.


**Supported file formats:**

- CSV
- JSON
- Parquet
- Avro
- ORC
- Text
- Binary files


### Real-time example:


**Consider an e-commerce application where customer order files arrive continuously in cloud storage.**

1. New customer order files arrive in AWS S3 or Azure Data Lake.
2. Auto Loader detects newly arriving files.
3. Spark reads and processes the files.
4. Data is cleaned and transformed.
5. Processed records are stored in a Delta Lake table.


### Architecture Flow

**1. Cloud Storage (Source)**

AWS S3 / Azure Data Lake / Google Cloud Storage

- Incoming customer order files.
- New CSV, JSON or Parquet files arrive continuously.

⬇️

**2. Databricks Auto Loader**

- Detects newly arriving files.
- Incrementally loads new files.
- Tracks processed files using checkpoint information.
- Supports schema inference and schema evolution.

⬇️

**3. Spark Structured Streaming**

- Filter unwanted records.
- Clean missing or duplicate data.
- Transform customer order data.
- Apply business logic.

⬇️

**4. Delta Lake (Sink)**

- Stores processed customer orders.
- Supports ACID transactions.
- Supports schema enforcement and time travel.
- Enables reliable analytical queries.



### Data Flow Summary:



| Component | Responsibility |
|---|---|
| Cloud Storage | Stores incoming files |
| Auto Loader | Detects and incrementally loads new files |
| Spark Structured Streaming | Processes and transforms data |
| Delta Lake | Stores processed data |




### Important Auto Loader Options:



| Option | Purpose |
|---|---|
| cloudFiles | Enables Auto Loader |
| cloudFiles.format | Specifies source file format |
| cloudFiles.schemaLocation | Stores inferred schema information |
| checkpointLocation | Tracks streaming progress and processed files |

In [0]:
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation",
            "/Volumes/workspace/default/data_volume/schema")
    .load("/Volumes/workspace/default/data_volume/orders")
)

(df.writeStream
    .format("delta")
    .option("checkpointLocation",
            "/Volumes/workspace/default/data_volume/checkpoint")
    .outputMode("append")
    .start("/Volumes/workspace/default/data_volume/processed_orders")
)


## 10. What are Streaming Tables?


- **Streaming Tables** are `Delta tables` in Databricks designed to `process and store data incrementally` from streaming sources.
- They `continuously or incrementally ingest new records` as they arrive, without reprocessing the entire dataset.
- They are commonly used in `Lakeflow Spark Declarative Pipelines` to build real-time data pipelines.


### Real-time example:


**Consider an e-commerce application receiving customer orders continuously.**

1. Customer places an order.
2. Order data arrives from Kafka.
3. Streaming Table processes incoming orders incrementally.
4. Invalid records are filtered.
5. Valid orders are stored in a Streaming Table.
6. Downstream analytics can consume the processed data.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_stream
AS
SELECT *
FROM STREAM read_files(
    '/mnt/orders/',
    format => 'csv',
    header => true
);


## 11. What are Materialized Views?


- **Materialized Views** are `database objects` used to `store the precomputed results` of a SQL query.
- Instead of executing the query every time, Databricks maintains` the query results and refreshes them` when the underlying source data changes.
- They are mainly used to `improve query performance` and `simplify reporting and analytics`.


### Real-time example:


**Consider an e-commerce company with millions of customer orders.**

**You need to calculate total sales for each region.**

**Without a Materialized View:**

- Every time a user runs the query, Spark processes the source data again.
- This can take more time for large datasets.

**With a Materialized View:**

- Databricks maintains the calculated sales results.
- Users can query the stored results instead of recalculating everything from scratch.


### SQL Example:


**Suppose we have an orders table with region and amount columns.**

In [0]:
%sql
CREATE MATERIALIZED VIEW regional_sales AS
SELECT
    region,
    SUM(amount) AS total_sales,
    COUNT(*) AS total_orders
FROM orders
GROUP BY region;


### To query the Materialized View:

In [0]:
%sql
SELECT *
FROM regional_sales;


## Difference Between Streaming Tables and Materialized Views


| Streaming Tables | Materialized Views |
|---|---|
| Designed for incremental data ingestion and processing. | Designed to store the results of a query. |
| Commonly processes new incoming records. | Maintains the results of a SQL query. |
| Useful for ingestion and streaming pipelines. | Useful for analytical queries and reporting. |
| Uses streaming semantics. | Supports incremental or full refresh. |
| Processes data incrementally as new data arrives. | Refreshes query results when source data changes. |
| Example: New customer orders ingestion. | Example: Total sales by region. |


## 12. What is DLT?


- **DLT (Delta Live Tables)** is a `Databricks framework` used to `build, manage, and automate` reliable data pipelines.
- It simplifies `managing data ingestion, transformations, dependencies, data quality expectations, and pipeline execution`.


### Real-time example:


**Consider an e-commerce company receiving customer orders from Kafka and cloud storage.**

DLT helps build an automated pipeline with three layers:
- **Bronze**: Store raw customer orders.
- **Silver**: Clean data, remove duplicates, and filter invalid records.
- **Gold**: Calculate total sales, revenue, and regional reports.


### DLT Architecture – Medallion Architecture:


**1. Data Sources (Source Layer)**

Kafka / AWS S3 / Azure Data Lake / CSV / JSON

- Receives `raw customer order data`.
- Supports batch and streaming data ingestion.

⬇️

**2. Bronze Layer (Raw Data)**

- Stores `raw customer order data`.
- Maintains the `original source data`.
- Preserves` data for auditing and reprocessing`.

Example: Raw customer orders.

⬇️

**3. Silver Layer (Cleaned Data)**

- `Cleans and validates` records.
- `Removes duplicate` records.
- Handles `missing and invalid` values.
- Applies `data quality` checks.

Example: Clean and validated customer orders.

⬇️

**4. Gold Layer (Business Data)**

- Performs `sales aggregations`.
- Calculates `total revenue and profit`.
- Creates `business-level summary tables`.
- Prepares data for reporting and analytics.

Example: Total sales by region and monthly revenue.

⬇️

**5. BI and Reporting Layer**

Power BI / SQL Dashboards

- Connects to `Gold tables`.
- Generates `business reports`.
- Displays` KPIs and analytical dashboards`.


### Layer Comparison:


| Layer | Purpose | Example |
|---|---|---|
| Bronze | Store raw data | Raw customer orders |
| Silver | Clean and validate data | Deduplicated orders |
| Gold | Business-level aggregation | Total sales by region |
| BI / Reporting | Visualize business insights | Power BI dashboard |




### Real-Time E-Commerce Example:


| Layer | Data Processing |
|---|---|
| Bronze | Store 15,000 raw customer orders |
| Silver | Remove duplicates and invalid orders |
| Gold | Calculate daily sales and regional revenue |
| Power BI | Display sales performance dashboard |


## What are the two Pipeline Modes in DLT?


1. Triggered Pipeline Mode
2. Continuous Pipeline Mode


### 1. Triggered Pipeline Mode:


- **Triggered pipeline mode** executes the `pipeline using the data available` when` the pipeline update starts`.
- After `processing the available data` and `refreshing the tables`, the pipeline automatically stops.


### Real-time example:


**An e-commerce company receives customer order files every hour.**

- At 10:00 AM, the pipeline starts.
- It processes all available order files.
- It cleans and transforms the data.
- It updates Bronze, Silver and Gold tables.
- The pipeline stops after completing the update.


### B. Continuous Pipeline Mode:


**Continuous mode** keeps the pipeline `running and processes new data as it arrives` to maintain fresh tables.


### Real-time example:


**An e-commerce company receives customer orders continuously from Kafka.**

- Customer places an order.
- New order data arrives in Kafka.
- The pipeline processes incoming records.
- Bronze, Silver and Gold tables are updated.
- Processing continues as new data arrives.


## 13. Streaming vs Batch Processing – Same Declarative Code?


In Databricks, the `same declarative pipeline code` can `support both batch and streaming processing` by changing `how the source data is read`.

- **Batch Processing:**

Processes a `fixed set of available data` at scheduled intervals.

- **Streaming Processing:** 

Processes `incoming data incrementally` as new records arrive.


## Batch Processing vs Streaming Processing – Architecture Comparison


### 1. Batch Processing Architecture:


**CSV Files (Source)**

- Receives` daily customer order files`.
- Stores `orders in CSV, JSON or Parquet format`.

⬇️

**Batch Pipeline**

- Reads `available data at scheduled intervals`.
- `Cleans, filters and transforms` records.
- `Processes data `in large batches.

⬇️

**Delta Lake (Sink)**

- Stores `processed customer orders`.
- Supports `reliable analytical queries`.


### Real-Time Example:


**An e-commerce company processes all customer orders at midnight to generate daily sales reports.**


### 2. Streaming Processing Architecture:


**Kafka (Source)**

- Receives `incoming customer orders continuously`.
- Publishes `order events to Kafka topics`.

⬇️

**Streaming Pipeline**

- Processes `incoming records incrementally`.
- `Filters, cleans and transforms` data.
- Processes data using `micro-batches`.

⬇️

**Delta Lake (Sink)**

- Stores `processed customer orders`.
- Makes updated `data available for analytics`.


### Real-Time Example: 


**An e-commerce company processes customer orders as they arrive and updates sales analytics continuously.**


### Architecture Comparison:


| Batch Processing | Streaming Processing |
|---|---|
| CSV Files | Kafka |
| Daily orders | Incoming orders |
| Batch Pipeline | Streaming Pipeline |
| Processes available data | Processes newly arriving data |
| Scheduled execution | Continuous or trigger-based execution |
| Higher processing delay | Lower processing delay |
| Delta Lake | Delta Lake |


### PySpark – Batch Processing:

In [0]:
batch_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/data/orders/")

result = batch_df \
    .groupBy("region") \
    .sum("amount")

display(result)


### PySpark – Streaming Processing:

In [0]:
stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(order_schema) \
    .load("/data/orders/")


### Apply the same transformation:

In [0]:
result = stream_df \
    .groupBy("region") \
    .sum("amount")


### Write streaming results:

In [0]:
result.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "/checkpoints/sales"
    ) \
    .start()


### 14. Watermark in PySpark DSL vs SQL


**Watermarking** in Spark Structured Streaming is used to `handle late-arriving data` and `control how long Spark maintains state for event-time operations`.


### Real-time example – E-commerce application


**Imagine an e-commerce application receiving customer orders.**

- Order A arrives at 10:00 AM.
- Order B arrives at 10:05 AM.
- Order C arrives at 10:02 AM but reaches Spark at 10:15 AM due to network delay.

**Order C is late-arriving data.**


**Suppose we configure a 10-minute watermark.**

**Spark allows late records within the watermark threshold and can eventually remove old aggregation state when it is safe to do so.**


### Event-time example:


1. 10:00 AM

   Order A event time

2. 10:05 AM

   Order B event time

3. 10:15 AM

   Order C arrives late with event time 10:02 AM

4. 10-minute watermark

   Controls late-data handling and state cleanup


### Watermark using PySpark DSL:

In [0]:
from pyspark.sql.functions import *

result = df \
    .withWatermark(
        "order_time",
        "10 minutes"
    ) \
    .groupBy(
        window("order_time", "5 minutes")
    ) \
    .agg(
        sum("amount").alias("total_sales")
    )


**Explanation:**


- **withWatermark()** defines the late-data threshold.
- **order_time** is the event-time column.
- **10 minutes** is the allowed lateness threshold.
- **window()** creates 5-minute time windows.
- **sum()** calculates total sales.


### B. Watermark using Databricks SQL:

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE sales_window
AS
SELECT
    window(order_time, '5 minutes') AS time_window,
    SUM(amount) AS total_sales
FROM STREAM(orders)
WATERMARK order_time DELAY OF INTERVAL 10 MINUTES
GROUP BY window(order_time, '5 minutes');


**Explanation:**


- **STREAM(orders)** reads the streaming source.
- **WATERMARK** defines the event-time threshold.
- **INTERVAL 10 MINUTES **specifies the delay.
- **GROUP BY** performs time-window aggregation.


## 15. Important Structured Streaming Interview Questions:



| No. | Interview Question | Key Concept |
|---:|---|---|
| 1 | What is Structured Streaming? | Incremental processing |
| 2 | What is the difference between Source and Sink? | Input and output |
| 3 | What is a Trigger in Structured Streaming? | Processing frequency |
| 4 | What are the three output modes? | Append, Update, Complete |
| 5 | Why do we use Watermarks? | Late data handling and state management |
| 6 | What is Windowing in Structured Streaming? | Time-based aggregation |
| 7 | What is Checkpointing? | Fault tolerance and recovery |
| 8 | How does Spark integrate with Kafka? | Real-time data ingestion |
| 9 | What is Auto Loader? | Incremental file ingestion |
| 10 | What is `cloudFiles`? | Auto Loader source format |
| 11 | What is `schemaLocation`? | Schema tracking and evolution |
| 12 | What is `checkpointLocation`? | Processing progress and state recovery |
| 13 | What is the difference between Streaming Tables and Materialized Views? | Incremental ingestion vs stored query results |
| 14 | What is the difference between Triggered and Continuous Pipeline Modes? | Cost vs data freshness |
| 15 | What is the difference between Batch and Streaming Processing? | Periodic vs incremental processing |
| 16 | What is the difference between Watermark in PySpark DSL and SQL? | DataFrame API vs SQL syntax |


### Additional Advanced Interview Questions:


| No. | Interview Question | Key Concept |
|---:|---|---|
| 17 | What happens if a streaming job fails? | Checkpoint recovery |
| 18 | What is the difference between Event Time and Processing Time? | Time semantics |
| 19 | What is State Management in Structured Streaming? | Maintaining aggregation state |
| 20 | What is the difference between Tumbling, Sliding and Session Windows? | Window operations |
| 21 | How do you handle duplicate events in streaming? | Deduplication |
| 22 | What is Exactly-Once Processing? | Processing guarantees |
| 23 | What is the difference between `readStream` and `read`? | Streaming vs batch reading |
| 24 | What is the difference between `writeStream` and `write`? | Streaming vs batch writing |
| 25 | How do you monitor a streaming query? | StreamingQueryProgress and monitoring |
| 26 | What is the role of Kafka partitions in Spark streaming? | Parallelism |
| 27 | What happens when late-arriving data exceeds the watermark? | Late-event handling |
| 28 | How do you optimize a Structured Streaming pipeline? | Performance optimization |
| 29 | What is the difference between `AvailableNow` and `Once` triggers? | Incremental multi-batch vs single-batch execution |
| 30 | Can Auto Loader process both batch and streaming data? | File ingestion flexibility |


### Most Important Topics to Prepare:


1. Structured Streaming architecture.
2. Sources and Sinks.
3. Trigger types.
4. Output modes.
5. Watermarks and late-arriving data.
6. Tumbling, Sliding and Session Windows.
7. Checkpointing and fault tolerance.
8. Kafka integration.
9. Auto Loader and cloudFiles.
10. Streaming Tables vs Materialized Views.
11. DLT / Lakeflow Spark Declarative Pipelines.
12. Batch vs Streaming architecture.